**Lab type:** debug

**Course:** ML101 — Intro to Machine Learning

**Lesson:** Clustering Analysis — Discovering Patterns Without Labels

**Task:** The AI-generated analysis below contains 3 bugs. For each bug: identify what is wrong, explain why the output is misleading, and write the corrected code in the fix cell.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

np.random.seed(42)
n = 300
# 3 natural clusters in income/spending space
income = np.concatenate([
    np.random.normal(30000, 5000, 100),
    np.random.normal(70000, 8000, 100),
    np.random.normal(120000, 10000, 100)
])
spending = np.concatenate([
    np.random.normal(200, 50, 100),
    np.random.normal(600, 100, 100),
    np.random.normal(1200, 150, 100)
])
age = np.random.normal(38, 10, n).clip(18, 70)
df = pd.DataFrame({'annual_income': income, 'monthly_spending': spending, 'age': age})
print(f"Dataset shape: {df.shape}")
print(df.describe().round(1))

## Step 1: Fitting KMeans

We apply KMeans clustering to segment customers by their financial behaviour. The dataset has three features: annual income, monthly spending, and age.

In [ ]:
# AI-generated — fitting KMeans to the raw customer features
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(df[['annual_income', 'monthly_spending', 'age']])  # <- Bug 1
print("Cluster sizes:", df['cluster'].value_counts().to_dict())
print("\nCluster centroids (raw feature space):")
print(pd.DataFrame(kmeans.cluster_centers_, columns=['annual_income', 'monthly_spending', 'age']).round(1))

**Bug 1 Investigation:** Run the cell above. The clusters form — but look at the describe() output from Setup. `annual_income` ranges from roughly 10,000 to 150,000 while `age` ranges from 18 to 70. What does this imply about which feature dominates the distance calculations in KMeans? Are the clusters reflecting all three features equally?

In [ ]:
# Fix Bug 1 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Step 2: Choosing the Number of Clusters

After correcting the scaling, we now decide how many clusters to use. The analyst scales the data first and then fits KMeans.

In [ ]:
# AI-generated — scaling features and fitting KMeans with a chosen k
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['annual_income', 'monthly_spending', 'age']])

kmeans3 = KMeans(n_clusters=3, random_state=42, n_init=10)  # <- Bug 2 (k chosen without analysis)
labels = kmeans3.fit_predict(X_scaled)
print(f"Silhouette score (k=3): {silhouette_score(X_scaled, labels):.3f}")
print("Used k=3 because it seemed reasonable.")

**Bug 2 Investigation:** Run the cell above. The silhouette score is printed but k was never justified — the comment admits it was a guess. What analysis should be run before committing to a value of k? What would that analysis look like for k values between 2 and 8?

In [ ]:
# Fix Bug 2 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Step 3: Selecting the Best k

The analyst computes inertia for a range of k values and uses it to programmatically select the best k.

In [ ]:
# AI-generated — computing inertia for k=2..8 and selecting best k
inertias = []
k_values = range(2, 9)
for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

best_k = list(k_values)[np.argmin(inertias)]  # <- Bug 3
print(f"Best k by lowest inertia: {best_k}")
print(f"Inertias: {[round(i, 1) for i in inertias]}")

**Bug 3 Investigation:** Run the cell above. Look at the inertia values printed. Do they increase or decrease as k grows? What does `np.argmin` return on a sequence that only ever decreases? Is the value of `best_k` meaningful? What metric properly balances cluster cohesion against separation?

In [ ]:
# Fix Bug 3 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

## Corrected Analysis

The cell below applies all three fixes in the correct order. Run it end-to-end to confirm the pipeline is now sound.

In [ ]:
# --- Corrected pipeline: all three bugs fixed ---

# Fix 1: standardise features BEFORE passing to KMeans
scaler_fixed = StandardScaler()
X_scaled_fixed = scaler_fixed.fit_transform(df[['annual_income', 'monthly_spending', 'age']])
print("Feature ranges after scaling (should all be ~[-3, 3]):")
print(pd.DataFrame(X_scaled_fixed, columns=['annual_income', 'monthly_spending', 'age']).describe().round(2))

# Fix 2 + Fix 3: choose k using silhouette score, not inertia argmin
k_values = range(2, 9)
inertias = []
silhouette_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled_fixed)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled_fixed, labels))

# Inertia always decreases — use it for elbow visualisation only
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_values), inertias, marker='o')
axes[0].set_title('Elbow Plot (inertia vs k)')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')

axes[1].plot(list(k_values), silhouette_scores, marker='o', color='orange')
axes[1].set_title('Silhouette Score vs k')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score')
plt.tight_layout()
plt.show()

best_k = list(k_values)[np.argmax(silhouette_scores)]  # highest silhouette = best k
print(f"\nBest k by silhouette score: {best_k}")
print(f"Silhouette scores: {[round(s, 3) for s in silhouette_scores]}")

# Final model with the data-driven k
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster_corrected'] = km_final.fit_predict(X_scaled_fixed)
print(f"\nFinal cluster sizes (k={best_k}):")
print(df['cluster_corrected'].value_counts().sort_index())